# Phase 1 — Data Preparation
## Dataset: GSE6575 — Gene Expression in Blood of Children with Autism Spectrum Disorder

**Platform**: Affymetrix HG-U133 Plus 2.0 (microarray)  
**Goal**: Prepare the gene expression matrix for machine learning — clean, transform, and scale the data.  
**Target variable**: Diagnosis group (Autism no regression / Autism with regression / General population / MR·DD)


In [ ]:
!pip install -r requirements.txt

In [ ]:
# ── Library imports ─────────────────────────────────────────────────────────
# GEOparse  : downloads and parses NCBI GEO datasets
# pandas    : data manipulation and DataFrame operations
# numpy     : numerical operations and array handling
# matplotlib/seaborn : visualization for data exploration
# sklearn   : StandardScaler for feature normalization, VarianceThreshold for filtering

import GEOparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries loaded successfully.')

---
## Step 0 — Load the Dataset from NCBI GEO

We use GEOparse to download GSE6575 directly from NCBI.  
`pivot_samples('VALUE')` restructures the data into a probe × sample matrix where each cell contains the normalized expression intensity.

In [ ]:
# Download and parse the GEO series (cached locally after first run)
gse = GEOparse.get_GEO('GSE6575', destdir='./', silent=True)

# pivot_samples builds a matrix: rows = probes (ID_REF), columns = sample IDs (GSM...)
expr = gse.pivot_samples('VALUE')

# Save raw matrix to CSV for reference
expr.to_csv('GSE6575.csv')

print('Raw expression matrix shape (probes × samples):', expr.shape)

In [ ]:
# ── Build the sample label series from GEO metadata ─────────────────────────
# Each GSM sample carries a 'characteristics_ch1' field in its metadata.
# We extract the diagnosis group (everything before the comma, e.g., 'Autism no regression').
# This series will become the target variable (y) once we transpose the matrix.

labels = {}
for gsm_id, gsm in gse.gsms.items():
    # characteristics_ch1 looks like: ['Autism no regression, Male']
    char = gsm.metadata['characteristics_ch1'][0]
    group = char.split(',')[0].strip()   # keep only the diagnosis part
    labels[gsm_id] = group

label_series = pd.Series(labels, name='condition')
print('Class distribution:')
print(label_series.value_counts().to_string())

---
## Section 1 — Data Description

Before any transformation, we examine the raw data to understand its structure, types, and content.  
This step is essential for making informed decisions in later cleaning and scaling steps.

In [ ]:
# ── 1.1 Shape and column types ───────────────────────────────────────────────
# The raw expr DataFrame has probes as rows and samples as columns.
# All 56 columns are float64 — continuous numerical gene expression intensities.

n_probes, n_samples = expr.shape
print(f'Number of probes  (rows)   : {n_probes:,}')
print(f'Number of samples (columns): {n_samples}')
print()
print('Column data types:')
print(expr.dtypes.value_counts())
print()
print('Index name (probe IDs):', expr.index.name)

In [ ]:
# ── 1.2 Target variable description ─────────────────────────────────────────
# The target is a categorical variable with 4 classes representing different
# neurological/developmental conditions studied in this ASD microarray experiment.

print('Target variable: diagnosis / condition group')
print('Type: categorical (4 classes)\n')

vc = label_series.value_counts()
print(vc.to_string())
print(f'\nTotal samples: {vc.sum()}')

# Visualise class distribution
fig, ax = plt.subplots(figsize=(8, 4))
vc.plot(kind='bar', ax=ax, color=sns.color_palette('muted', 4))
ax.set_title('Sample Count per Diagnosis Group')
ax.set_xlabel('Condition')
ax.set_ylabel('Number of samples')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 1.3 First rows (head) ────────────────────────────────────────────────────
# Displays the first 5 probes across all 56 samples.
# Each value is a normalized expression intensity (Affymetrix microarray).

print('First 5 rows of the raw expression matrix:')
expr.head()

In [ ]:
# ── 1.4 Last rows (tail) ─────────────────────────────────────────────────────
print('Last 5 rows of the raw expression matrix:')
expr.tail()

In [ ]:
# ── 1.5 Descriptive statistics ───────────────────────────────────────────────
# Computing stats across all probes per sample to detect any anomalous samples.
# We check the global range to confirm the data is purely numerical before scaling.

desc = expr.describe().T   # transpose so each row is a sample
print('Per-sample summary statistics (first 5 samples):')
print(desc.head().to_string())
print()
print(f'Global min  : {expr.values.min():.4f}')
print(f'Global max  : {expr.values.max():.2f}')
print(f'Global mean : {expr.values.mean():.4f}')
print(f'Global std  : {expr.values.std():.4f}')
print()
print('Observation: max (1188) >> mean (0.88) → values are already numerical and on a comparable scale.')
print('Since all expression values are numerical, we scale them directly with StandardScaler (Section 3).')

---
## Section 2 — Data Cleaning

We perform cleaning in the following order:

| Step | Operation | Justification |
|------|-----------|---------------|
| 2.1 | Transpose matrix | ML convention: samples as rows, features as columns |
| 2.2 | Check missing values | NaN values break most ML algorithms |
| 2.3 | Remove near-zero-variance probes | Constant features carry no signal and inflate dimensionality |
| 2.4 | Check for duplicate samples | Duplicates cause data leakage between train/test splits |

In [ ]:
# ── 2.1 Transpose: probes × samples → samples × probes ──────────────────────
# GEOparse returns data with probes as rows and samples as columns.
# Machine learning expects the opposite: rows = observations (samples),
# columns = features (probes). We transpose and attach the class labels.

X = expr.T.copy()   # shape: (56 samples, 54675 probes)
y = label_series.loc[X.index]   # align labels to the same sample order

print('After transposing:')
print(f'  X shape : {X.shape}  (samples × probes)')
print(f'  y shape : {y.shape}  (sample labels)')
print()
print('First 3 sample IDs:', list(X.index[:3]))
print('Their labels      :', list(y.iloc[:3]))

In [ ]:
# ── 2.2 Missing value analysis ───────────────────────────────────────────────
# Missing values (NaN) must be identified before any transformation.
# Even a single NaN in a column propagates through scaling and model training.

total_values   = X.size
missing_values = X.isna().sum().sum()
missing_cols   = X.isna().any().sum()   # probes that have at least one NaN

print(f'Total cells         : {total_values:,}')
print(f'Missing values      : {missing_values}')
print(f'Probes with any NaN : {missing_cols}')
print()
if missing_values == 0:
    print('✓ No missing values — no imputation required.')
else:
    print('Missing values detected — see next cell for handling strategy.')

In [ ]:
# ── 2.3 Near-zero-variance probe removal ─────────────────────────────────────
# Probes whose expression barely changes across all 56 samples carry no
# discriminative information for classification. Keeping them wastes memory
# and can hurt distance-based algorithms by adding meaningless dimensions.
#
# Strategy: compute the standard deviation of each probe across all samples.
# Probes with std < 0.01 (in original linear scale) are effectively constant.
# We drop them before scaling to avoid amplifying noise.

probe_std = X.std(axis=0)   # std of each probe across the 56 samples

STD_THRESHOLD = 0.01   # probes with std below this are considered near-constant
low_var_mask  = probe_std < STD_THRESHOLD

print(f'Near-zero-variance probes (std < {STD_THRESHOLD}): {low_var_mask.sum()}')
print(f'Probes retained: {(~low_var_mask).sum()}')

# Drop the near-zero-variance probes
X_clean = X.loc[:, ~low_var_mask].copy()

print(f'\nX shape after removing low-variance probes: {X_clean.shape}')

In [ ]:
# ── 2.4 Duplicate sample check ───────────────────────────────────────────────
# Duplicate rows (identical expression profiles) can cause data leakage if
# the same sample appears in both train and test sets during cross-validation.

n_dupes = X_clean.duplicated().sum()
print(f'Duplicate samples: {n_dupes}')
if n_dupes == 0:
    print('✓ No duplicate samples found.')
else:
    X_clean = X_clean.drop_duplicates()
    y = y.loc[X_clean.index]
    print(f'Removed {n_dupes} duplicate(s). New shape: {X_clean.shape}')

In [ ]:
# ── 2.5 Cleaning summary ─────────────────────────────────────────────────────
print('=== Data Cleaning Summary ===')
print(f'  Original shape   : {X.shape}')
print(f'  After cleaning   : {X_clean.shape}')
print(f'  Probes removed   : {X.shape[1] - X_clean.shape[1]}  (near-zero-variance)')
print(f'  Samples removed  : {X.shape[0] - X_clean.shape[0]}  (duplicates)')
print(f'  Missing values   : 0  (none found)')
print(f'  Target y shape   : {y.shape}')

---
## Section 3 — Scaling & Transformation

| Step | Method | Why |
|------|--------|-----|
| 3.1 | **StandardScaler (z-score)** | The dataset is already numerical, so we scale it directly after cleaning. StandardScaler centers each probe to mean=0 and scales it to std=1, making probes comparable for machine learning algorithms (PCA, SVM, k-NN, logistic regression, etc.). |


In [ ]:
# ── 3.1 StandardScaler directly ─────────────────────────────────────────────
# The dataset is already numerical, so we apply z-score scaling directly.
# StandardScaler centers each probe to mean=0 and scales it to std=1.
# This follows the course method for numerical data scaling.
#
# We fit the scaler only on the feature matrix (X_clean), never on the labels (y).

scaler = StandardScaler()

X_scaled_array = scaler.fit_transform(X_clean)

X_scaled = pd.DataFrame(
    X_scaled_array,
    index=X_clean.index,
    columns=X_clean.columns
)

print('After StandardScaler:')
print(f'  Mean: {X_scaled.values.mean():.6f}')
print(f'  Std : {X_scaled.values.std():.6f}')
print(f'  Shape: {X_scaled.shape}')


In [ ]:
# ── Visualise the effect of StandardScaler ───────────────────────────────────
# We sample 5000 values at random for plotting (avoids memory issues with 3M+ values).

rng = np.random.default_rng(42)
flat_raw    = X_clean.values.ravel()
flat_scaled = X_scaled.values.ravel()
idx = rng.choice(len(flat_raw), size=5000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(flat_raw[idx], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Raw expression values\n(original scale)')
axes[0].set_xlabel('Intensity'); axes[0].set_ylabel('Frequency')

axes[1].hist(flat_scaled[idx], bins=60, color='mediumseagreen', edgecolor='white')
axes[1].set_title('After StandardScaler\n(centered at 0, unit variance)')
axes[1].set_xlabel('z-score'); axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
# ── Compare one sample's expression profile before and after scaling ────────
# After scaling, the sample distribution should be centred near 0.

sample_id = X_clean.index[0]   # pick the first sample as an example

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_clean.loc[sample_id], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title(f'Raw values — sample {sample_id}')
axes[0].set_xlabel('Intensity')
axes[0].set_ylabel('Frequency')

axes[1].hist(X_scaled.loc[sample_id], bins=60, color='mediumseagreen', edgecolor='white')
axes[1].set_title(f'After StandardScaler — sample {sample_id}')
axes[1].set_xlabel('z-score')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.3 Final dataset overview ───────────────────────────────────────────────
# Combine scaled features with the target label to produce the final dataset.

df_final = X_scaled.copy()
df_final.insert(0, 'condition', y)   # insert label as first column

print('Final prepared dataset:')
print(f'  Shape         : {df_final.shape}  (samples × [label + probes])')
print(f'  Columns [0]   : condition (target)')
print(f'  Columns [1..] : {X_scaled.shape[1]:,} gene expression probes (z-scores)')
print()
print('Label distribution:')
print(df_final['condition'].value_counts().to_string())

print()
print('First 3 rows (label + first 5 probes):')
df_final.iloc[:3, :6]

In [ ]:
# ── Save final prepared dataset to CSV ───────────────────────────────────────
# Saving both X (features) and y (labels) so Phase 2 can load them directly.

X_scaled.to_csv('X_prepared.csv')       # feature matrix (samples × probes)
y.to_csv('y_labels.csv', header=True)   # label series
df_final.to_csv('dataset_final.csv')    # combined (label + features)

print('Saved:')
print('  X_prepared.csv  — scaled feature matrix')
print('  y_labels.csv    — sample labels')
print('  dataset_final.csv — full prepared dataset')

---
## Summary of Phase 1

| Step | Action | Justification |
|------|--------|---------------|
| Load | GEOparse → `pivot_samples('VALUE')` | Programmatic download, no manual file handling |
| Labels | Extracted from `characteristics_ch1` metadata | Ground truth is embedded in sample metadata |
| Transpose | Probes × Samples → **Samples × Probes** | ML convention: rows = observations |
| Missing values | None found | No action required |
| Near-zero-variance | Removed 45 probes (std < 0.01) | Constant features add noise, no signal |
| Duplicates | None found | No action required |
| StandardScaler | Applied directly per-probe (column-wise) on the cleaned numerical data | Centers and scales each gene to mean=0, std=1; required for PCA, SVM, k-NN |
| **Final shape** | **56 samples × 54,630 probes** | Ready for Phase 2 (modeling) |

**Target variable**: `condition` — 4-class categorical label
- Autism no regression (17 samples)
- Autism with regression (18 samples)
- General population (12 samples)
- Mental retardation / developmental delay (9 samples)
